# BGL Log Data Exploration

Exploring the [BGL supercomputer log dataset](https://github.com/logpai/loghub/tree/master/BGL) to build a clean `(text, label)` DataFrame for training a log-anomaly classifier. Along the way: parse a tricky raw format, surface data-quality issues, and figure out how to handle massive duplication and class imbalance.

## 1. Setup

In [1]:
import gc

import pandas as pd

## 2. Loading & Parsing the Raw Log File

The BGL log has 10 logical fields per line separated by single spaces. The 10th field (`Content`) is free text that contains spaces itself, so a naive CSV parse with `sep=' '` would shred the message across many columns.

**Approach:**

1. Read the file as a single column by passing `sep='\x07'` (a character that doesn't appear in the file) to `pd.read_csv`. Each line becomes one row in one column.
2. Vectorize the split with `Series.str.split(n=9, expand=True)`. The `n=9` caps the split at 9 separators, leaving everything after the 9th field as one chunk - which is what `Content` should be.

Faster than `readlines()` + list comprehension because pandas reads in C, and uses less memory because we hold one intermediate DataFrame instead of three Python lists.

In [2]:
raw_data_path = "../data/raw/BGL.log"
columns = [
    "Label",
    "Timestamp",
    "Date",
    "Node",
    "Time",
    "NodeRepeat",
    "Type",
    "Component",
    "Level",
    "Content",
]

# Read as single column (no real separator in file)
raw = pd.read_csv(
    raw_data_path,
    sep="\x07",
    header=None,
    names=["line"],
    engine="python",
    encoding="utf-8",
    on_bad_lines="skip",
)

# Vectorized split into 10 columns
df = raw["line"].str.split(n=9, expand=True)
df.columns = columns
print(df.head())

  Label   Timestamp        Date                 Node  \
0     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
1     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
2     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
3     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   
4     -  1117838570  2005.06.03  R02-M1-N0-C:J12-U11   

                         Time           NodeRepeat Type Component Level  \
0  2005-06-03-15.42.50.363779  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
1  2005-06-03-15.42.50.527847  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
2  2005-06-03-15.42.50.675872  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
3  2005-06-03-15.42.50.823719  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   
4  2005-06-03-15.42.50.982731  R02-M1-N0-C:J12-U11  RAS    KERNEL  INFO   

                                    Content  
0  instruction cache parity error corrected  
1  instruction cache parity error corrected  
2  instruction cache parity error corrected  
3  instruction cache parity 

In [3]:
# Free the single-column intermediate; df has everything we need.
del raw

## 3. Initial Structural Inspection

Before any cleaning, looking at the structured columns to:

- Confirm we got the expected ~4.7M rows
- Validate column types and memory footprint
- **Surface unexpected values** in fields that should have small, fixed vocabularies (`Level`, `Component`, `Type`). Garbage values in these columns are a signal that some rows didn't parse correctly.

In [4]:
print("DataFrame Info:")
print(df.info())

print(f"\nUnique Labels: {df['Label'].value_counts()}")
print(f"\nUnique Components: {df['Component'].value_counts()}")
print(f"\nUnique Severities: {df['Level'].value_counts()}")

DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 4747963 entries, 0 to 4747962
Data columns (total 10 columns):
 #   Column      Dtype
---  ------      -----
 0   Label       str  
 1   Timestamp   str  
 2   Date        str  
 3   Node        str  
 4   Time        str  
 5   NodeRepeat  str  
 6   Type        str  
 7   Component   str  
 8   Level       str  
 9   Content     str  
dtypes: str(10)
memory usage: 362.2 MB
None

Unique Labels: Label
-            4399503
KERNDTLB      152734
KERNSTOR       63491
APPSEV         49651
KERNMNTF       31531
KERNTERM       23338
KERNREC         6145
APPREAD         5983
KERNRTSP        3983
APPRES          2370
APPUNAV         2048
APPTO           1991
KERNMICRO       1503
APPOUT           816
KERNMNT          720
APPBUSY          512
KERNMC           342
APPCHILD         320
KERNSOCK         209
KERNPOW          192
LINKIAP          166
APPALLOC         144
KERNSERV          94
MASABNORM         37
LINKDISC          24
KERNPAN        

### Findings

- ~4.75M rows parsed; memory footprint ~362 MB (string dtypes are heavy).
- `Label` shows the expected `-` (normal) plus 30+ alert categories with a long-tail distribution. Several categories have only 1-3 examples - too few for multi-class learning; binary is the realistic target.
- **`Level` contains 4 unexpected values** (`Kill`, `single`, `microseconds`, `0x00544eb8,`) totaling 316 rows
- **`Component` shows matching garbage** (`FATAL`, `a`, `0`, `iar`)

These garbage values are not valid Level or Component entries - they're **misaligned fields from malformed rows** where the original log line didn't follow the expected 10-field structure. `str.split` placed wrong data in wrong columns.

**Decision:** filter to canonical `Level` values to drop these rows.

## 4. Cleaning Malformed Rows

Filter to the 6 canonical Level values defined in the BGL schema: `INFO`, `FATAL`, `ERROR`, `WARNING`, `SEVERE`, `FAILURE`. This drops ~316 rows out of 4.7M (~0.007%) - negligible for downstream work.

In [5]:
canonical_levels = {"INFO", "FATAL", "ERROR", "WARNING", "SEVERE", "FAILURE"}

before = len(df)
df = df[df["Level"].isin(canonical_levels)].reset_index(drop=True)
dropped = before - len(df)

print(f"Dropped {dropped} malformed rows ({dropped / before * 100:.4f}% of total)")
print(f"Remaining: {len(df):,}")

Dropped 316 malformed rows (0.0067% of total)
Remaining: 4,747,647


In [6]:
# Verify Level and Component are now clean
print("Level:")
print(df["Level"].value_counts())
print("\nComponent:")
print(df["Component"].value_counts())

Level:
Level
INFO       3735813
FATAL       855195
ERROR       112355
WARNING      23357
SEVERE       19213
FAILURE       1714
Name: count, dtype: int64

Component:
Component
KERNEL       4324651
APP           228536
DISCOVERY      97172
MMCS           88930
HARDWARE        5148
MONITOR         1681
LINKCARD        1170
CMCS             211
BGLMASTER        145
SERV_NET           3
Name: count, dtype: int64


## 5. Building the Target DataFrame

The classifier consumes exactly two columns: `text` and `label`.

**Decisions:**

- **`text` = Content only.** Including `Level` (which contains `FATAL`/`SEVERE`/`ERROR` etc.) would leak the label into the features. A model trained on Level + Content would learn `if Level == FATAL → anomaly` - that's effectively a regex, not a classifier. Excluding Level forces the model to learn from message semantics.
- **`label` = binarized** (0 = normal where `Label == '-'`, 1 = anomaly otherwise). The 30+ alert categories are heavily imbalanced; multi-class is a possible extension later, not the baseline.

In [7]:
# Binarize the Label column
df["is_anomaly"] = (df["Label"] != "-").astype(int)
print(f"\nClass Distribution:\n{df['is_anomaly'].value_counts()}")
print(f"\nClass Distribution Normalized:\n{df['is_anomaly'].value_counts(normalize=True)}")

anomaly_ratio = df["is_anomaly"].mean()
print(f"\nAnomaly Ratio: {anomaly_ratio:.4f} ({anomaly_ratio*100:.2f}%)")


Class Distribution:
is_anomaly
0    4399187
1     348460
Name: count, dtype: int64

Class Distribution Normalized:
is_anomaly
0    0.926604
1    0.073396
Name: proportion, dtype: float64

Anomaly Ratio: 0.0734 (7.34%)


In [8]:
# Project down to the two columns the model will consume
data = df[["Content", "is_anomaly"]].rename(columns={"Content": "text", "is_anomaly": "label"})
print(data.head())
print(data.shape)

                                       text  label
0  instruction cache parity error corrected      0
1  instruction cache parity error corrected      0
2  instruction cache parity error corrected      0
3  instruction cache parity error corrected      0
4  instruction cache parity error corrected      0
(4747647, 2)


## 6. Exploratory Data Analysis

Beyond class balance (the headline number for any anomaly-detection problem), check:

- **Missing values** - parser artifacts often manifest as NaN
- **Duplicates** - log datasets often have heavy duplication; the same template repeats thousands of times. The duplicate count drives the train/test splitting strategy.
- **Text length distribution** - informs vectorizer choices (max features, n-gram range)
- **Memory footprint**
- **Side-by-side anomaly vs. normal samples** - eyeball whether the classes are textually distinguishable

In [9]:
# Missing values
print(f"\nMissing Values:{data.isnull().sum()}")

# Exact duplicate text lines
dup_count = data["text"].duplicated().sum()
print(f"\nDuplicate text values: {dup_count:,} ({dup_count/len(data)*100:.2f}%)")

# Unique text values vs total rows
print(f"\nUnique text values: {data['text'].nunique():,} out of {len(data):,}")

# Text length distribution
print("\nText Length Distribution:")
print(data["text"].str.len().describe())

# Memory footprint
print(f"\nMemory: {data.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Side-by-side sample: 5 anomalies vs 5 normal
print("\nAnomaly samples:")
print(data[data["label"] == 1]["text"].sample(5, random_state=42).to_list())
print("\nNormal samples:")
print(data[data["label"] == 0]["text"].sample(5, random_state=42).to_list())


Missing Values:text     34470
label        0
dtype: int64

Duplicate text values: 4,389,601 (92.46%)

Unique text values: 358,045 out of 4,747,647

Text Length Distribution:
count    4.713177e+06
mean     4.920158e+01
std      5.204476e+01
min      1.000000e+00
25%      2.100000e+01
50%      2.600000e+01
75%      4.500000e+01
max      8.380000e+02
Name: text, dtype: float64

Memory: 501.9 MB

Anomaly samples:
['data TLB error interrupt', 'data TLB error interrupt', 'data TLB error interrupt', 'data TLB error interrupt', 'data TLB error interrupt']

Normal samples:
['generating core.31149', nan, 'Ido chip status changed: FF:F2:9F:16:B8:24:00:0D:60:E9:47:DB ip=10.5.1.70 v=13 t=4 status=M Tue Sep 20 12:11:48 PDT 2005', 'iar 00105e78 dear 02f5a65c', 'generating core.57264']


### Findings

- **34,470 NaN text values** - rows where the original line had fewer than 10 whitespace-separated tokens. Parser artifact; will drop next.
- **92.46% duplicates** - only ~358K unique text values across 4.7M rows. The dataset is effectively a 358K-row dataset with each row repeated ~13x on average.
- **Text length:** median 26 chars, max 838. Short messages dominate. TF-IDF will work fine.
- **Anomaly sample** (random_state=42, 5 picks) all returned the same string `'data TLB error interrupt'`. With a fixed seed, getting 5 identical strings from 348K positives is essentially zero by chance - one template massively dominates the positive class.
- **Normal samples** show real diversity: chip status changes, core dumps, register dumps.

### Critical implication: do not random-split

With 92% duplicates, a naive `train_test_split` puts identical strings in both train and test. F1 will look near-perfect and mean nothing - the model isn't generalizing, it's looking up. We need to split on **unique texts**, not raw rows.

More generally: split on the unit of generalization, not the unit of observation. Same rule applies to user-level splits (recommenders), patient-level splits (medical ML), session-level splits (clickstream), and time-based splits (anything temporal).

## 7. Null Handling

Drop the 34K NaN text rows. These came from malformed lines that didn't have a `Content` field after the 9th split.

Re-run duplicate analysis afterward: `duplicated()` treats NaN as equal to NaN, so the 34K NaNs were previously counted as one giant duplicate cluster, slightly inflating the 92.46% figure.

In [10]:
data = data.dropna(subset=["text"]).reset_index(drop=True)

dup_count = data["text"].duplicated().sum()
unique_count = data["text"].nunique()

print(f"Rows after dropping NA: {len(data):,}")
print(f"Duplicate rows: {dup_count:,} ({dup_count/len(data)*100:.2f}%)")
print(f"Unique rows: {unique_count:,}")

Rows after dropping NA: 4,713,177
Duplicate rows: 4,355,132 (92.40%)
Unique rows: 358,045


### Result

Duplicate percentage moved 92.46% → 92.40% after NaN removal. The headline story is unchanged: the dataset has ~358K genuinely unique messages.

## 8. Label Consistency Check

Before deduplicating by text, check whether any text value appears with both labels (0 and 1). If it does, deduplication loses information - whichever row pandas keeps first determines the surviving label, which is effectively arbitrary.

Same input mapping to different outputs is usually a signal of mislabeling, ambiguous taxonomy, or temporal drift in the data.

In [11]:
# Does any text appear with both labels?
label_per_text = data.groupby("text")["label"].nunique()
inconsistent = (label_per_text > 1).sum()
print(f"Texts with inconsistent labels: {inconsistent}\n")

if inconsistent > 0:
    problem_texts = label_per_text[label_per_text > 1].index[:5]
    for t in problem_texts:
        counts = data[data["text"] == t]["label"].value_counts().to_dict()
        print(f"  {t[:80]!r:<82} -> {counts}")

Texts with inconsistent labels: 3

  'Microloader Assertion'                                                            -> {1: 1503, 0: 1}
  'data storage interrupt'                                                           -> {1: 63491, 0: 2}
  'program interrupt'                                                                -> {0: 3662, 1: 5}


### Findings

Only 3 texts show inconsistent labels:

| Text                       | Anomaly count | Normal count |
|----------------------------|--------------:|-------------:|
| Microloader Assertion      | 1,503         | 1            |
| data storage interrupt     | 63,491        | 2            |
| program interrupt          | 5             | 3,662        |

These are label noise - likely 1-2 mislabeled rows for the first two, and `program interrupt` is overwhelmingly normal with 5 anomaly outliers.

**Decision:** let `drop_duplicates` keep whichever row pandas encounters first. The impact on the ~358K unique row set is < 0.001%. A stricter approach would be majority-vote or dropping these texts entirely - not worth the complexity here.

## 9. Deduplication & Final Class Balance

Build the pipeline summary table and compute the anomaly ratio on **unique** texts. The unique-row ratio is the number that drives modeling decisions - the raw 7.34% is misleading because of the massive duplication.

In [16]:
unique_data = data.drop_duplicates(subset=["text"]).reset_index(drop=True)

# Stage 1: cleaning pipeline (rows actually dropped)
cleaning = pd.DataFrame(
    [
        {"stage": "Raw lines parsed", "rows": 4_747_963},
        {"stage": "After Level canonical filter", "rows": 4_747_647},
        {"stage": "After dropna(text)", "rows": len(data)},
    ]
)
cleaning["pct_of_raw"] = (cleaning["rows"] / 4_747_963 * 100).round(2)

# Stage 2: deduplication (informational - duplicates still exist in production)
dedup = pd.DataFrame(
    [
        {"view": "Cleaned rows (production distribution)", "rows": len(data)},
        {"view": "Unique text rows (training universe)", "rows": len(unique_data)},
    ]
)

print("Cleaning pipeline:")
print(cleaning.to_string(index=False))
print("\nDeduplication view:")
print(dedup.to_string(index=False))

print("\nRaw-row class distribution:")
print(data["label"].value_counts(normalize=True).round(4))

print("\nUnique-row class distribution:")
print(unique_data["label"].value_counts(normalize=True).round(4))

print(f"\nRaw anomaly ratio:    {data['label'].mean()*100:.2f}%")
print(f"Unique anomaly ratio: {unique_data['label'].mean()*100:.2f}%")

Cleaning pipeline:
                       stage    rows  pct_of_raw
            Raw lines parsed 4747963      100.00
After Level canonical filter 4747647       99.99
          After dropna(text) 4713177       99.27

Deduplication view:
                                  view    rows
Cleaned rows (production distribution) 4713177
  Unique text rows (training universe)  358045

Raw-row class distribution:
label
0    0.9261
1    0.0739
Name: proportion, dtype: float64

Unique-row class distribution:
label
0    0.8631
1    0.1369
Name: proportion, dtype: float64

Raw anomaly ratio:    7.39%
Unique anomaly ratio: 13.69%


### Findings

**Raw anomaly ratio: 7.39%. Unique anomaly ratio: 13.69%.**

The unique-row ratio is nearly 2x the raw rate. This is a real finding worth understanding: **normal logs are more templated than anomaly logs**. Normal operation has a limited vocabulary of repeating messages; failures generate more diverse text.

This reframes the modeling problem - we're not learning from 4.7M examples, we're learning from ~358K. The model's job is closer to "memorize the anomaly templates and flag anything matching them" than "deeply understand log semantics."

## 10. Memory Cleanup

`df` is no longer needed - everything downstream cares about is in `unique_data`. Free the ~400 MB.

In [13]:
del df
gc.collect()

0

## 11. Summary

### Data pipeline

| Stage                          | Rows        | % of raw |
|--------------------------------|------------:|---------:|
| Raw lines parsed               | 4,747,963   | 100.00   |
| After Level canonical filter   | 4,747,647   |  99.99   |
| After dropna(text)             | 4,713,177   |  99.27   |
| Unique text rows               |   358,045   |   7.54   |

### Decisions made during exploration

| Decision                              | Rationale                                                |
|---------------------------------------|----------------------------------------------------------|
| `text` = Content (Level excluded)     | Prevents Level→Label leakage (FATAL correlates with anomaly) |
| Binary labels (0/1)                   | Multi-class brutally imbalanced; baseline is normal-vs-anomaly |
| Drop malformed rows via Level filter  | 316 rows / 4.7M, removes column-shift artifacts          |
| Drop NaN text rows                    | 34K parser artifacts                                     |
| Train on unique texts, not raw rows   | 92% duplicates would leak under random split             |

### Class imbalance and metric choice

Unique-row distribution: **86.31% normal / 13.69% anomaly**.

Accuracy is not a useful metric here - a "predict always 0" model gets 86% accuracy without learning anything. In production it would never flag a real anomaly: high metric, useless model.

Better metrics:

- **Precision** = of predicted anomalies, what fraction were real? *(when you cry wolf, how often is there a wolf?)*
- **Recall** = of real anomalies, what fraction did we catch? *(how many wolves did we miss?)*
- **F1** = harmonic mean of the two, the number that punishes cheating on either.

For log anomaly detection, **recall usually matters more than precision** - missing a real failure costs downtime. False alarms cost analyst time. Operations teams generally prefer investigating a few false positives over missing a real outage.

### Open questions

- Is ~358K unique examples enough for the model to generalize, or will it essentially memorize all the templates?
- What happens to F1 if the model is also tested on the **original repeated distribution** (the world the API will actually see in production)?
- How does TF-IDF with character n-grams compare to word n-grams on this data?
- Aggregate F1 is the wrong number to optimize. The real test is recall on anomaly templates that are rare or absent in training — the failures the model has never encountered. That's the difference between a model that generalizes and a lookup table that pretends to.